In [0]:
%sh pwd

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Actual Management P&L"

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Flat Actual Management P&L"

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Budget Management P&L"

In [0]:
from pyspark.sql import functions as F

df_final_py = df_final_py.withColumn('year', col('year')+1)


join_keys = ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 
             'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 
             'sub_group', 'management_details_total', 'type', 'year', 'account_type', 
             'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 
             'country_code', 'group', 'management_group']

# Get the non-key (metric) columns from each df
actual_metrics = [c for c in df_final_actual.columns if c not in join_keys]
py_metrics     = [c for c in df_final_py.columns if c not in join_keys]
budget_metrics = [c for c in df_final_budget.columns if c not in join_keys]

# Add missing metric columns as null so all 3 dfs have the same schema
all_metrics = list(set(actual_metrics + py_metrics + budget_metrics))

def add_missing_cols(df, all_cols, existing_cols):
    for col in all_cols:
        if col not in existing_cols:
            df = df.withColumn(col, F.lit(None).cast("double"))  # adjust type if needed
    return df

df_actual_full = add_missing_cols(df_final_actual, all_metrics, actual_metrics)
df_py_full     = add_missing_cols(df_final_py,     all_metrics, py_metrics)
df_budget_full = add_missing_cols(df_final_budget, all_metrics, budget_metrics)

# Union all three and group by keys, taking first non-null value per metric column
df_temp = (
    df_actual_full
    .unionByName(df_py_full)
    .unionByName(df_budget_full)
    .groupBy(join_keys)
    .agg(*[F.first(c, ignorenulls=True).alias(c) for c in all_metrics])
)
# df_temp.display()

df_final = df_temp.fillna({'amount': 0, 'py_amount': 0, 'budget_amount': 0})


df_final_selected = df_final.select(
    'city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'Detail/Total',
    'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 
    'account_type', 'store_open_date2', 'brand_id', 'company_id', 'parent_company', 'country_code',
    'group', 'management_group', 'year', 'month', 'netsuite_location_name', 'mapped_name', 'amount', 'budget_amount', 'py_amount'
).orderBy(
            "parent_company", "company_id", "brand_id", 
            "netsuite_location_name", 'year', 'month', 'management_sort_order'
        )

In [0]:
df_final_selected.filter(col('netsuite_location_name').isNotNull()).filter(col('year')==2026).filter(col("brand_id") == "ALBAIK").display()

In [0]:
%sql
use catalog fq_dev_pnl_catalog;

In [0]:
def enrich_data(df):
    # df = spark.read.table('default.postman_response_january')
    df_exploded = df.select(explode('results').alias('result')).select('result.*')

    # ✅ Reverse sign when accountType == 'Income'
    df_exploded = df_exploded.withColumn(
        "amount",
        when(col("accountType") == "Income", -col("amount"))
        .when(col("accountName") == "75704 Talabat- Commission Discount", -col("amount"))
        .when(col("accountName") == "75713 Noon- Commission Discount", -col("amount"))
        .otherwise(col("amount"))
    ).withColumn(
        "year", col('year').cast('int')
    )

    df_brand_ho_rows = spark.read.table('fq_dev_pnl_catalog.bronze.brand_ho_allocation_cost')

    df_exploded_keys = df_exploded.select(
        col("year"), 
        col("month"), 
        col("location").alias("netsuite_location_name")
    ).distinct()


    df_brand_ho_rows_filtered = df_brand_ho_rows.join(
        df_exploded_keys,
        (df_brand_ho_rows["year"] == df_exploded_keys["year"]) &
        (df_brand_ho_rows["month"] == df_exploded_keys["month"]) &
        (df_brand_ho_rows["location"] == df_exploded_keys["netsuite_location_name"]),
        "inner"
    ).select(df_brand_ho_rows["*"])

    # Perform union with filtered data
    df_exploded = df_exploded.union(df_brand_ho_rows_filtered)

    df_coa_master = spark.read.table("bronze.dim_coa_master")
    # df_coa_master = to_snake_case_df(df_coa_master)
    df_location_master = spark.read.table("bronze.dim_location_master")
    # df_location_master = to_snake_case_df(df_location_master)
    # df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


    # for column in df_coa_master.columns:
    #     if dict(df_coa_master.dtypes)[column] == 'string':
    #         df_coa_master = df_coa_master.withColumn(column, trim(col(column)))

    df_all_masters = df_exploded.join(
            df_coa_master, 
            df_coa_master["account_number"].cast("string") == df_exploded["accountNo"], 
            'inner'
        ).join(
            df_location_master,
            col("location") == df_location_master.netsuite_location_name,
            'left'
        )

    # Exclude summary "Total" accounts that duplicate management group totals
    df_all_masters_filtered = df_all_masters.filter(
        ~col("account_name").startswith("Total ")
    )

    return final_df(df_all_masters_filtered)

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Formula & Functions Management P&L"

In [0]:
df = spark.read.option('multiline', False).format('json').load('abfss://staging@fqadfstoragedev.dfs.core.windows.net/FoodQuest/Netsuite/GL_Report/ALBAIK/2026/JAN/gl_report.json')
df_final_actual = enrich_data(df)
df_final_all = df_final_all(df_final_actual)
df_final_all.display()

In [0]:
def df_final_all(df_final_actual):
    df_final_actual_keys = df_final_actual.select(
            col("year"), 
            col("month"), 
            col("netsuite_location_name")
        ).distinct()
    # df_final_actual_keys.display()

    df_management_pnl = spark.read.table('fq_dev_pnl_catalog.silver.management_pnl')

    df_final_py = df_management_pnl.join(
        df_final_actual_keys,
        (df_management_pnl["year"] == (df_final_actual_keys["year"]-1)) &
        (df_management_pnl["month"] == df_final_actual_keys["month"]) &
        (df_management_pnl["netsuite_location_name"] == df_final_actual_keys["netsuite_location_name"]),
        "inner"
    ).select(df_management_pnl["*"]).drop("budget_amount").drop("py_amount")

    # df_final_py.display()

    df_pnl_budget_flat_data = spark.read.table('fq_dev_pnl_catalog.silver.pnl_budget_flat_data')

    df_final_budget = df_pnl_budget_flat_data.join(
        df_final_actual_keys,
        (df_pnl_budget_flat_data["year"] == df_final_actual_keys["year"]) &
        (df_pnl_budget_flat_data["month"] == df_final_actual_keys["month"]) &
        (df_pnl_budget_flat_data["netsuite_location_name"] == df_final_actual_keys["netsuite_location_name"]),
        "inner"
    ).select(df_pnl_budget_flat_data["*"])

    # df_final_budget.display()

    from pyspark.sql import functions as F

    df_final_py = df_final_py.withColumn("year", col('year')+1).withColumnRenamed('amount', 'py_amount')



    join_keys = ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 
                'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 
                'sub_group', 'management_details_total', 'type', 'year', 'account_type', 
                'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 
                'country_code', 'group', 'management_group']

    # Get the non-key (metric) columns from each df
    actual_metrics = [c for c in df_final_actual.columns if c not in join_keys]
    py_metrics     = [c for c in df_final_py.columns if c not in join_keys]
    budget_metrics = [c for c in df_final_budget.columns if c not in join_keys]

    # Add missing metric columns as null so all 3 dfs have the same schema
    all_metrics = list(set(actual_metrics + py_metrics + budget_metrics))

    def add_missing_cols(df, all_cols, existing_cols):
        for col in all_cols:
            if col not in existing_cols:
                df = df.withColumn(col, F.lit(None).cast("double"))  # adjust type if needed
        return df

    df_actual_full = add_missing_cols(df_final_actual, all_metrics, actual_metrics)
    df_py_full     = add_missing_cols(df_final_py,     all_metrics, py_metrics)
    df_budget_full = add_missing_cols(df_final_budget, all_metrics, budget_metrics)

    # Union all three and group by keys, taking first non-null value per metric column
    df_temp = (
        df_actual_full
        .unionByName(df_py_full)
        .unionByName(df_budget_full)
        .groupBy(join_keys)
        .agg(*[F.first(c, ignorenulls=True).alias(c) for c in all_metrics])
    )
    # df_temp.display()

    df_final = df_temp.fillna({'amount': 0, 'py_amount': 0, 'budget_amount': 0})


    return df_final.select(
        'city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'Detail/Total',
        'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 
        'account_type', 'store_open_date2', 'brand_id', 'company_id', 'parent_company', 'country_code',
        'group', 'management_group', 'year', 'month', 'netsuite_location_name', 'mapped_name', 'amount', 'budget_amount', 'py_amount'
    ).orderBy(
                "parent_company", "company_id", "brand_id", 
                "netsuite_location_name", 'year', 'month', 'management_sort_order'
            )

In [0]:
df_final_selected.display()